# RTSE · Colab 全流程（数据准备 → 训练 → 导出评测）

三个原 notebook 合并成这一个。**合并的唯一目的是省时间**：
Colab 的 `/content` 每个会话都会清空，原来每开一个 notebook 就要重跑一遍
「解压代码包 + 解压 20GB 语料」，一次约 20 分钟，整条链路要付三次；
现在初始化只跑一次。

## 怎么用

1. 先跑 **初始化** 的 4 个 cell（挂载 → 配置 → 安装代码 → 自检）；
2. 然后按 PART 1 → 2 → 3 顺序往下跑。

三个 PART 之间用 `═══` 分割线隔开，每段开头都写了它负责什么。
**PART 2 的闸门不过就不要跑 PART 3**——导出一个连干净输入都会破坏的模型没有意义。

## Drive 目录约定

```
MyDrive/Audio AI/RTSE/
├── colab_upload/     ← 你上传的：rtse-colab.zip + 本 notebook
├── colab_outputs/    ← 所有产物，要下载就整个下这一个目录
│   ├── checkpoints/  models/  testsets/  logs/
│   └── manifest.json  training_gates.json  colab_metrics.json  rtse_handoff.zip
└── archives/         ← 已下好的约 20GB 语料缓存，**位置不动**
```

`archives/` 刻意留在原处：搬走会让下载标记全部失效，20GB 得重下一遍。

---

# ═══════════════════════════════════════════════════════════════════
# 初始化（只需跑一次）
# ═══════════════════════════════════════════════════════════════════


In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# V1 已停用 QUICK_TEST：两套受控集都依赖 DNS 留出噪声/RIR。
# 小规模跑通请用 SMOKE_RUN=True；QUICK_TEST 必须保持 False。
QUICK_TEST = False

# ── 小规模跑通模式 ─────────────────────────────────────────────────────
# True = 大幅缩小**用量**与**训练时长**，验证「数据→训练→导出→评测」整条链路。
# 缩的不是下载量（DNS 分片是最小单位，该下多少还是多少），而是"用多少条"和"跑多少轮"：
#   受控集   81 格 × 1 = 81 条/套（正式 5/格 = 405）
#   真实 CER 3 个时长桶 × 10 = 30 条（正式 300）
#   噪声分类 抽 400 条做平稳性判决（正式 4000）
#   训练     2000 样本/epoch × 3 epoch，只跑 crn-nano（正式 20000 × 60，两档）
# 跑通之后改成 False 再跑正式版。
SMOKE_RUN = True

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'
assert not QUICK_TEST, 'V1 不支持 QUICK_TEST；请改用 SMOKE_RUN=True 做小规模闭环'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

# ── 上传区与产物区分开 ─────────────────────────────────────────────────
# 你上传的东西都在 colab_upload/，Colab 产出的东西都在 colab_outputs/，
# 要下载结果就整个下 colab_outputs/ 这一个目录，不用在 Drive 根目录里挑。
UPLOAD_DIR = f'{DRIVE}/colab_upload'
OUT = f'{DRIVE}/colab_outputs'

CKPT_DIR = f'{OUT}/checkpoints'     # 训练断点，每 epoch 保存
MODEL_DIR = f'{OUT}/models'         # 导出的 ONNX
TESTSETS_DIR = f'{OUT}/testsets'    # V1 三套职责分离的固定测试集
MANIFEST = f'{OUT}/manifest.json'          # 数据清单
GATES = f'{OUT}/training_gates.json'       # 训练闸门结果
COLAB_METRICS = f'{OUT}/colab_metrics.json'  # Colab 侧指标（含 PESQ）
DNS_QUALITY_DIR = f'{TESTSETS_DIR}/dns_objective'
AISHELL_CER_DIR = f'{TESTSETS_DIR}/aishell_controlled'
WENET_REAL_DIR = f'{TESTSETS_DIR}/wenetspeech_real'
LOG_DIR = f'{OUT}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, UPLOAD_DIR, OUT,
          CKPT_DIR, MODEL_DIR, TESTSETS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {UPLOAD_DIR}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}   ← 位置不动，别搬')
print(f'  语料解压目标        {DATA}')
print(f'  ── 以下全在 colab_outputs/，下载就下这一个目录 ──')
print(f'  数据清单            {MANIFEST}')
print(f'  三套固定测试集      {TESTSETS_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 训练/质量 + AISHELL 受控 CER + WenetSpeech 真实 CER)"}')
print(f'  规模      {"⚡ 小规模跑通（SMOKE_RUN=True，结果不作数）" if SMOKE_RUN else "正式规模"}')
print()

!df -h /content | tail -1


In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 Drive 的 colab_upload/ 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{UPLOAD_DIR}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/colab_upload/ 下的全部内容传到 Drive 的 {UPLOAD_DIR}/ 下。\n'
    f'该目录下现有：{sorted(os.listdir(UPLOAD_DIR)) if os.path.isdir(UPLOAD_DIR) else "（目录不存在）"}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
# **重新导入前必须把已加载的 rtse 从 sys.modules 里清掉。**
# 上面的 unzip 换的是磁盘上的文件，而 `import rtse` 对**已经导入过**的模块是空操作：
# 同一个 Colab 会话里重跑本 cell，磁盘上是新代码、内存里跑的还是旧的。
# 这个症状极具迷惑性——报错的行号来自旧文件，跟你手里的新文件对不上号，
# 会让人以为"包没传上去"而反复重传（见 docs/ISSUES.md I-13 / I-28）。
for _m in [m for m in list(sys.modules) if m == 'rtse' or m.startswith('rtse.')]:
    del sys.modules[_m]
import rtse
# 自证：把**实际加载的文件路径和改动时间**打出来。
# "我改的代码到底有没有在跑"必须是可观测的事实，不能靠推断（I-28）。
print('已加载 rtse ←', rtse.__file__)
print('           改动时间',
      time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(rtse.__file__))),
      '| 若这个时间不是你刚打包的时刻，说明跑的还是旧代码')
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。Colab 上的 STFT 与本地哪怕差一点，训练出来的模型拿回本地就会
# 掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

---

# ═══════════════════════════════════════════════════════════════════
# PART 1 · 数据准备
# ═══════════════════════════════════════════════════════════════════

下载 DNS5 / AISHELL-1 / WenetSpeech，建清单、按平稳性给噪声分组，
生成三套职责分离的 V1 测试集。

**耗时最长的一段**：首次要下约 20 GB（之后走 `archives/` 缓存），
每个新会话都要重新解压到 `/content`（Drive 是 FUSE，训练直接读会很慢）。

## 1. 下载 DNS Challenge 语料

分片是**独立的** `.tar.bz2`，可以只下需要的几片 —— 这是能做小规模子集的前提。
（DNS5 的 clean speech 是 `split` 切片，必须全部下载才能拼接解压，用不了。）

下载支持断点续传（`wget -c`）；已下好的会跳过，Colab 断线重连后重跑不会从头再来。

In [ ]:
DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

# lbzip2：多线程解压 bzip2。噪声分片是 .tar.bz2，单线程解 5 GB 要 ~9 分钟，
# 而 Colab 每个会话都得重解一遍（/content 会被清空），这是最大的一块固定开销。
# 装不上也能跑，只是退回单线程。
!apt-get -qq install -y lbzip2 > /dev/null 2>&1 || true
print('lbzip2:', '可用（解压将并行）' if shutil.which('lbzip2') else '不可用（退回单线程 bzip2）')

def fetch_dns(name, blob_path, expect_min_wavs=50, partial_ok=False):
    """下载 → 解压一个 DNS 分片。带下载/解压双标记，支持断点续传与跨会话复用。

    Args:
        partial_ok: 该分片是 `split` 切片（干净语音），解压到末尾必然报
            "Unexpected EOF"。设 True 时忽略这个错误——只要解出足够多的
            完整文件就算成功。校验靠**实际解出的 wav 数量**，不靠 tar 的返回码。

    校验方式统一是"解压后递归扫到的 wav 数量"，而不是断言某个具体子目录名——
    实测 DNS5 语音解出来的路径是 `mnt/dnsv5/clean/read_speech/...`，
    嵌套好几层且没写在官方文档里。下游 scan() 本来就是递归扫描，不关心层数。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    # 解压标记里记下**当时解出的文件数**和格式版本。只有"新格式 + 文件数对得上"
    # 才认为可信。旧格式（空文件 touch 出来的）可能是 I-26 修复前写下的——
    # 那时截断的解压也会被当成成功（实测有分片只解出 1156/7739 却被记成已完成），
    # 一律重解一遍。这样**修复能追溯地清理掉此前留下的坏状态**，
    # 而不是只对将来生效、让已经写坏的标记永远把分片挡在门外。
    if os.path.exists(ex_mark):
        try:
            mark = json.loads(Path(ex_mark).read_text())
        except Exception:
            mark = None
        n_now = count_wavs()
        if isinstance(mark, dict) and mark.get('v', 0) >= 2 and mark.get('n') == n_now:
            print(f'[skip  ] {name} 已解压（{n_now} 个 wav）'); return True
        why = ('标记是旧格式，无法确认当时是否解压完整'
               if mark is None else
               f'文件数与标记不符（记录 {mark.get("n")}，实际 {n_now}）')
        print(f'[recheck] {name} {why} —— 重解一遍')

    def free_gb(path):
        try:
            return shutil.disk_usage(path).free / 1e9
        except Exception:
            return float('nan')

    # 下载标记里记下**当时的字节数**，缓存命中时核对一遍。
    # 只看"标记在不在"是不够的：压缩包可能被截断、被覆盖、或下到一半留下残file。
    # 旧格式（`touch` 出来的空文件）没有这个信息，就地升级成新格式，
    # **不重新下载**——那是 20 GB，代价太大，而且现有包已逐个核对过与服务器一致。
    cached_ok = False
    if os.path.exists(dl_mark) and os.path.exists(archive):
        size_now = os.path.getsize(archive)
        try:
            rec = json.loads(Path(dl_mark).read_text())
        except Exception:
            rec = None
        if isinstance(rec, dict) and rec.get('bytes') is not None:
            if rec['bytes'] == size_now:
                cached_ok = True
                print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，大小与记录一致）')
            else:
                # 用**字节数**报差异，不用 GB：两个只差几 MB 的数字，
                # 格式化成 GB 会显示成一模一样，等于没给信息。
                print(f'[FAIL  ] {name} 压缩包大小与记录不符 —— 文件被改动或截断。')
                print(f'         记录 {rec["bytes"]:,} 字节，实际 {size_now:,} 字节'
                      f'（差 {size_now - rec["bytes"]:+,}）')
                print(f'         删掉 {archive} 和 {dl_mark} 后重跑，或直接重跑让 -c 续传。')
                return False
        else:
            cached_ok = True
            Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size_now}))
            print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，标记已补记大小）')
    if not cached_ok:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        wlog = f'{ARCHIVE_DIR}/.{name}.wget.log'
        # 进度**必须可见**：几个 GB 的下载如果一声不吭，中途看起来和卡死没区别。
        # 同时又要留下日志，失败时才说得出原因，所以用 tee 兼顾两者。
        # `-T 60` 的超时在长连接上很容易触发，但 `-c` 会断点续传——
        # 重跑本 cell 就能接着下，**不要删掉已下载的部分**。
        # 走 bash 是因为要取 PIPESTATUS（Colab 的 /bin/sh 是 dash，不支持）。
        cmd = (f'wget --progress=dot:giga -c -T 60 -O {shq(archive)} {shq(url)} '
               f'2>&1 | tee {shq(wlog)}; exit ${{PIPESTATUS[0]}}')
        rc = subprocess.run(['bash', '-c', cmd]).returncode
        size = os.path.getsize(archive) if os.path.exists(archive) else 0
        if rc != 0 or size < 1e6:
            # **不要直接断言"blob 路径变了"**——那只是众多可能之一，而且是最不可能的那个。
            # 实测最常见的是**空间不足**：hybrid 模式下归档写在 Drive 上，
            # 本批归档合计约 20 GB，而 Drive 免费版只有 15 GB。
            # 先把证据摆出来（wget 原话 + 两个卷的剩余空间），再让人去判断。
            print(f'[FAIL  ] {name} 下载未完成（wget 退出码 {rc}，已落盘 {size/1e9:.2f} GB）')
            print('         ⚠️ 多数情况下这只是超时中断，**已下载的部分是有效的**：'
                  '直接重跑本 cell，-c 会接着下。')
            tail = subprocess.run(f'tail -n 3 {shq(wlog)}', shell=True,
                                  capture_output=True, text=True).stdout.strip()
            if tail:
                print('         wget: ' + tail.replace(chr(10), chr(10) + '         '))
            print(f'         剩余空间：归档卷 {free_gb(ARCHIVE_DIR):.1f} GB，'
                  f'解压卷 {free_gb(DATA):.1f} GB')
            print('         本批归档合计约需 20 GB（语音 5.2 + 噪声 14.2 + IR 0.3）。')
            print('         排查顺序：① 上面两个卷的剩余空间够不够；'
                  '② Drive 挂载是否还活着（ls 一下 DRIVE 目录）；')
            print('         ③ 都正常再去 https://github.com/microsoft/DNS-Challenge '
                  '核对 blob 路径（这一项本地 HEAD 实测过 200，最不可能）。')
            return False
        Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size}))

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    # 压缩格式**按文件头判断，不能看扩展名**：切片档叫 `read_speech.tgz.partaa`，
    # 结尾是 `.partaa` 而不是 `.tgz`，用 endswith 会判成 bzip2；GNU tar 拿 -j 去解
    # gzip 流会直接报 "is not a bzip2 file" 且一个文件都不解出。
    # Colab 上实际踩过：语音分片解出 0 个 wav（见 docs/ISSUES.md I-25）。
    with open(archive, 'rb') as fh:
        magic = fh.read(2)
    if magic == b'\x1f\x8b':
        flag, prog = 'xzf', None
    else:
        # **bzip2 解压是单线程 CPU 瓶颈**：5 GB 分片实测约 9 分钟，全程只吃一个核。
        # lbzip2 能对**任意** bzip2 流做多线程解压（pbzip2 只能并行它自己压出来的），
        # Colab 两个 vCPU 大致能快一倍。装不上就退回单线程，不影响正确性。
        prog = 'lbzip2' if shutil.which('lbzip2') else None
        flag = 'xf' if prog else 'xjf'
    decomp = f'--use-compress-program={prog} ' if prog else ''
    # 切片档忽略 tar 的非零返回码（末尾必然 EOF），靠文件数判断成败。
    # stderr 落盘而不是丢进 /dev/null——失败时要能说出为什么失败。
    log = f'{out_dir}/.untar.log'
    with open(log, 'w') as lf:
        rc_tar = subprocess.run(f'tar {decomp}-{flag} {shq(archive)} -C {shq(out_dir)}',
                                shell=True, stderr=lf).returncode
    n = count_wavs()

    def tar_err():
        t = subprocess.run(f'tail -n 5 {shq(log)}', shell=True,
                           capture_output=True, text=True).stdout.strip()
        return ('\n         tar: ' + t.replace(chr(10), chr(10) + '         ')) if t else ''

    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav（用 {flag} 解，文件头 {magic!r}）。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB')
        print(f'         删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False
    # 切片档解到末尾必然 EOF，退出码非零属正常；**其余压缩包退出码非零意味着解压被截断**，
    # 而此时 wav 数很可能仍然过线，于是被当成成功放过去。
    # 实测踩过：噪声分片只解出 1156/7739 个文件却报 [done]，训练数据悄悄少了 85%。
    if rc_tar != 0 and not partial_ok:
        print(f'[FAIL  ] {name} 解压未完成：tar 退出码 {rc_tar}，只解出 {n} 个 wav。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB —— 空间不足是最常见原因。')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive); Path(dl_mark).unlink(missing_ok=True)
    # 记下文件数，下次才能判断"这份解压结果还是不是完整的那一份"
    Path(ex_mark).write_text(json.dumps({'v': 2, 'n': n}))
    note = '（切片档，尾部 EOF 属正常）' if partial_ok else ''
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒 {note}')
    return True


def dns_shards():
    """按配置生成要下载的分片清单 (名字, blob 路径)。"""
    # 语音是 split 切片，按字母序 partaa/partab/...；每片约 5.24 GB ≈ 19 小时。
    # ⚠️ **只有 partaa 能单独解压**：gzip 头只在第一片里，partab 及之后都是
    # 裸的流中段，单独拿去 tar 解必然失败（文件头实测是 b'\n\xc5' 这类随机字节）。
    # 要更多语音只能把所有片下全再 `cat *.part* | tar xz`，那是 110 GB。
    assert N_SPEECH_SHARDS == 1, (
        'N_SPEECH_SHARDS 只能是 1：DNS5 语音是 split 切片，gzip 头只在 partaa 里，'
        '后续切片无法独立解压。需要更多数据请改用别的数据源，或准备 110 GB 下全部切片。')
    parts = ['aa', 'ab', 'ac', 'ad', 'ae']
    sp = [(f'dns_speech_{p}', f'Track1_Headset/read_speech.tgz.part{p}')
          for p in parts[:N_SPEECH_SHARDS]]
    nz = [(f'dns_noise_audioset_{i:03d}',
           f'noise_fullband/datasets_fullband.noise_fullband.audioset_{i:03d}.tar.bz2')
          for i in range(N_AUDIOSET_SHARDS)]
    nz += [(f'dns_noise_freesound_{i:03d}',
            f'noise_fullband/datasets_fullband.noise_fullband.freesound_{i:03d}.tar.bz2')
           for i in range(N_FREESOUND_SHARDS)]
    # ⚠️ IR 分片在 blob 根目录下，**没有** `impulse_responses/` 前缀
    # （语音和噪声分片才有目录前缀）。写错会 404 —— 本地 HEAD 请求实测确认过。
    ir = [('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2')]
    return sp, nz, ir

sp_shards, nz_shards, ir_shards = dns_shards()
print(f'计划下载：语音 {len(sp_shards)} 片、噪声 {len(nz_shards)} 片、IR {len(ir_shards)} 片')
free = shutil.disk_usage(DATA).free / 1e9
print(f'解压卷可用 {free:.1f} GB（估计需要 40~60 GB）')

if QUICK_TEST:
    print('\n[QUICK_TEST] 跳过 DNS 下载，只用 WenetSpeech 验证链路')
    results = {}
else:
    t0 = time.time()
    results = {}
    for n, b in sp_shards: results[n] = fetch_dns(n, b, expect_min_wavs=500, partial_ok=True)
    for n, b in nz_shards: results[n] = fetch_dns(n, b, expect_min_wavs=100)
    for n, b in ir_shards: results[n] = fetch_dns(n, b, expect_min_wavs=50)
    print(f'\n总耗时 {(time.time()-t0)/60:.1f} 分钟   结果: {results}')
    assert all(results.values()), '有分片没就绪，看上面的 FAIL 信息'

!df -h {shq(DATA)} | tail -1

## 2. 下载两套中文评测语料

- **AISHELL-1 test**：安静室内、高保真麦克风录制，用于可控中文 CER。这里使用
  Hugging Face 的 test-only parquet 镜像第一片（约 398 MB，约 2400 条），不用下载
  OpenSLR 15 GB 全量包；它足够生成本项目的小规模固定测试集。
- **WenetSpeech test_meeting**：真实会议录音，用于真实场景 CER。保持原始音频，
  不再叠加 DNS 噪声或 RIR。


In [ ]:
import pandas as pd

AISHELL_URL = ('https://huggingface.co/datasets/TwinkStart/AISHELL-1/resolve/'
               'refs%2Fconvert%2Fparquet/default/test/0000.parquet')
WNS_URL = ('https://huggingface.co/datasets/lmms-lab/WenetSpeech/resolve/main/'
           'data/test_meeting-00000-of-00001.parquet')

def fetch_parquet(name, url):
    path = f'{ARCHIVE_DIR}/{name}.parquet'
    if not os.path.exists(path):
        print(f'[get   ] {name} ← {url}')
        rc = os.system(f'wget --show-progress -c -T 60 -O {shq(path)} {shq(url)}')
        assert rc == 0 and os.path.getsize(path) > 1e6, f'{name} 下载失败；直接重跑可断点续传'
    else:
        print(f'[cached] {name} ({os.path.getsize(path)/1e6:.0f} MB)')
    return path

aishell_parquet = fetch_parquet('aishell1_test_0000', AISHELL_URL)
wns_parquet = fetch_parquet('wenetspeech_test_meeting', WNS_URL)
aishell = pd.read_parquet(aishell_parquet)
wns = pd.read_parquet(wns_parquet)

print(f'\nAISHELL-1 子片: {len(aishell)} 条，列 {list(aishell.columns)}')
print(f'WenetSpeech meeting: {len(wns)} 条，列 {list(wns.columns)}')


## 3. 建立文件清单 + 噪声按平稳性分类

语音按**说话人**划分（避免同一人同时出现在训练和测试，模型靠记音色作弊）。
噪声用 `rtse.dsp.stationarity` 逐个算平稳性并分成两组 —— 这是后面
"DSP 在稳态噪声上够用、在非稳态上失效"这条对照能不能立住的前提。

In [ ]:
import random
from rtse.dsp.stationarity import stationarity_features, DEFAULT_DR_THRESHOLD_DB
from rtse.audio.io import read_audio
from tqdm.auto import tqdm

def scan(root, exts=('.wav', '.flac')):
    root = Path(root)
    if not root.exists(): return []
    return sorted(str(p) for p in root.rglob('*') if p.suffix.lower() in exts)

speech_all, noise_all, rir_all = [], [], []
if not QUICK_TEST:
    for n, _ in sp_shards: speech_all += scan(f'{DATA}/{n}')
    for n, _ in nz_shards: noise_all += scan(f'{DATA}/{n}')
    for n, _ in ir_shards: rir_all += scan(f'{DATA}/{n}')

print(f'语音 {len(speech_all):>7} 条')
print(f'噪声 {len(noise_all):>7} 条')
print(f'RIR  {len(rir_all):>7} 条')
if not QUICK_TEST:
    assert speech_all and noise_all and rir_all, '有类别扫不到文件，检查上一步解压结果'

In [ ]:
# ── 噪声平稳性分类 ─────────────────────────────────────────────────────
# 全量算太慢（几万个文件），抽样一部分做分类；每个文件只读前 10 秒就够判断。
MAX_NOISE_TO_CLASSIFY = 400 if SMOKE_RUN else 4000
rnd = random.Random(42)
noise_pool = noise_all[:]
rnd.shuffle(noise_pool)
noise_pool = noise_pool[:MAX_NOISE_TO_CLASSIFY]

stationary, nonstationary, skipped = [], [], 0
for p in tqdm(noise_pool, desc='噪声平稳性分类'):
    try:
        y = read_audio(p)[:16000 * 10]
    except Exception:
        skipped += 1; continue
    y_ac = y - y.mean()
    if not np.all(np.isfinite(y)) or np.mean(y_ac ** 2) <= 1e-12:
        # 常量/静音文件没有可定义的 SNR；以前会混进 stationary 组，最终生成 inf。
        skipped += 1; continue
    f = stationarity_features(y_ac)
    if f.n_frames < 32:           # 太短，无法可靠判断
        skipped += 1; continue
    (stationary if f.detrended_dynamic_range_db < DEFAULT_DR_THRESHOLD_DB
     else nonstationary).append(p)

print(f'\n稳态   {len(stationary):>5} 条')
print(f'非稳态 {len(nonstationary):>5} 条')
print(f'跳过   {skipped:>5} 条（太短或读取失败）')
print(f'\n门限 {DEFAULT_DR_THRESHOLD_DB} dB —— 这个值是在合成噪声上标定的，'
      f'真实录音分布更连续，如果两组比例悬殊（比如 9:1）就该调它。')
assert stationary and nonstationary, '有一组是空的，门限需要重新标定'

In [ ]:
# ── 划分 train/test ────────────────────────────────────────────────────
# 语音按**说话人**划分，不是按文件随机划分：按文件随机会让同一个人同时出现在
# 训练集和验证集里，模型能靠"记住这个人的音色"作弊，指标虚高。
#
# ⚠️ DNS5 的文件名形如 `book_00000_chp_0009_reader_06709_0_seg_1_seg1.wav`，
# 说话人 id 藏在 `reader_XXXXX` 这一段里。**不能用 `stem.split('_')[0]`**——
# 那会对每个文件都返回 "book"，所有语音归成一个说话人，划分彻底失效。
# 这个 bug 本地用 Range 请求取前 80 MB 实测抓到过（231 个文件切出 47 位说话人）。
import re

def speaker_of(p):
    m = re.search(r'reader_(\d+)', str(p))
    return m.group(1) if m else Path(p).stem

spk_probe = sorted({speaker_of(p) for p in speech_all[:2000]})
print(f'说话人 id 提取自检：前 2000 个文件切出 {len(spk_probe)} 位说话人，'
      f'样例 {spk_probe[:5]}')
assert 1 < len(spk_probe) < len(speech_all[:2000]) * 0.9, (
    '说话人提取异常：要么全归成一个人（正则没匹配上），要么几乎每个文件一个人'
    '（文件名格式变了）。两种情况都会让划分失去意义，必须先修这里。'
)

spk = sorted({speaker_of(p) for p in speech_all})
random.Random(20260807).shuffle(spk)
n_val = max(2, len(spk) // 10)
spk_val = set(spk[:n_val])
split = {'train': [], 'val': []}
for p in speech_all:
    split['val' if speaker_of(p) in spk_val else 'train'].append(p)
print(f'说话人 {len(spk)} 位 → train {len(spk)-n_val} / val {n_val}')
for k, v in split.items():
    print(f'  语音 {k:>5}: {len(v):>7} 条')

# 噪声与 RIR 也划分：测试用的必须是训练没见过的
def split_list(xs, frac=0.2, seed=42):
    xs = xs[:]; random.Random(seed).shuffle(xs)
    cut = max(1, int(len(xs) * frac))
    return xs[cut:], xs[:cut]           # (train, test)

st_train, st_test = split_list(stationary)
ns_train, ns_test = split_list(nonstationary)
rir_train, rir_test = split_list(rir_all)
print(f'  稳态噪声  train/test: {len(st_train)}/{len(st_test)}')
print(f'  非稳态噪声 train/test: {len(ns_train)}/{len(ns_test)}')
print(f'  真实 RIR  train/test: {len(rir_train)}/{len(rir_test)}')

manifest = {
    'version': 'dns_wenetspeech',
    'quick_test': QUICK_TEST,
    'data_dir': DATA,
    'speech': split,
    'noise_train': st_train + ns_train,
    'noise_test': st_test + ns_test,
    'noise_stationary_train': st_train, 'noise_stationary_test': st_test,
    'noise_nonstationary_train': ns_train, 'noise_nonstationary_test': ns_test,
    'rir_train': rir_train, 'rir_test': rir_test,
    'stationarity_threshold_db': DEFAULT_DR_THRESHOLD_DB,
}
Path(MANIFEST).write_text(
    json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
print(f'\n清单已写入 {MANIFEST}')

## 4. 生成三套固定测试集

两套受控集使用同一个80格噪声/RIR矩阵，并额外加入1格 clean→clean 无害性测试：

`SNR 5档 × 噪声 2类 × RIR来源 2类 × RT60 4档 = 80格，另加 identity = 81格`

- `SMOKE_RUN=True`：每格1条，每套81条；WenetSpeech按短/中/长各10条。
- 正式：每格5条，每套405条；WenetSpeech按短/中/长各100条。

受控集的 `clean` 字段表示**带混响、无加性噪声目标**；WenetSpeech 真实集没有
`clean` 字段，也不会产生虚假的“clean 上界”。


In [ ]:
import io
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm
from rtse.data.benchmarks import BenchmarkSource

MIN_SEG_SEC, MAX_SEG_SEC = 3.0, 15.0

def decode_embedded(rec):
    # 解码 HF parquet 的嵌入音频，统一成 16 kHz 单声道 float64。
    a = rec['audio']
    data, sr = sf.read(io.BytesIO(a['bytes']), dtype='float64', always_2d=False)
    if data.ndim == 2:
        data = data.mean(axis=1)
    if sr != 16000:
        import soxr
        data = soxr.resample(data, sr, 16000, quality='VHQ')
    return np.asarray(data, dtype=np.float64)

def parquet_sources(frame, id_fields, limit):
    out = []
    for i in range(len(frame)):
        rec = frame.iloc[i]
        try:
            audio = decode_embedded(rec)
        except Exception:
            continue
        text = str(rec.get('text') or rec.get('sentence') or '').strip()
        duration = audio.size / 16000
        if not (MIN_SEG_SEC <= duration <= MAX_SEG_SEC) or len(text.replace(' ', '')) < 5:
            continue
        source_id = next((str(rec.get(k)) for k in id_fields if rec.get(k) is not None), str(i))
        out.append(BenchmarkSource(source_id, audio, text))
        if len(out) >= limit:
            break
    return out

need_controlled = 100 if SMOKE_RUN else 600
need_real = 60 if SMOKE_RUN else 500
aishell_sources = parquet_sources(aishell, ('name', 'utt_id'), need_controlled)
wns_sources = parquet_sources(wns, ('utt_id', 'aid'), need_real)

dns_sources = []
if not QUICK_TEST:
    dns_candidates = manifest['speech']['val'][:]
    random.Random(20260818).shuffle(dns_candidates)
    for path in dns_candidates:
        try:
            audio = read_audio(path)
        except Exception:
            continue
        duration = audio.size / 16000
        if MIN_SEG_SEC <= duration <= MAX_SEG_SEC:
            dns_sources.append(BenchmarkSource(Path(path).stem, audio))
        if len(dns_sources) >= need_controlled:
            break

print(f'DNS 留出源语音 {len(dns_sources)} 条')
print(f'AISHELL 受控源语音 {len(aishell_sources)} 条')
print(f'WenetSpeech 真实源语音 {len(wns_sources)} 条')
assert aishell_sources and wns_sources
if not QUICK_TEST:
    assert dns_sources


In [ ]:
from rtse.data.benchmarks import (
    RT60_BINS, build_real_rir_buckets,
    generate_controlled_benchmark, generate_real_cer_benchmark,
)

PER_CELL = 1 if SMOKE_RUN else 5
WENET_PER_BUCKET = 10 if SMOKE_RUN else 100

if not QUICK_TEST:
    print('按实测 RT60 给真实 RIR 分桶…')
    real_rir_buckets = build_real_rir_buckets(
        rir_test,
        min_per_bucket=3 if SMOKE_RUN else 20,
        max_scan=3000 if SMOKE_RUN else 20000,
    )
    print('真实 RIR 桶:', {k: len(v) for k, v in real_rir_buckets.items()})
    noise_by_kind = {'stationary': st_test, 'nonstationary': ns_test}

    dns_index = generate_controlled_benchmark(
        dns_sources, noise_by_kind, real_rir_buckets, DNS_QUALITY_DIR,
        dataset_id='dns5_objective_v1', purpose='objective_audio_quality',
        source_dataset='DNS5 held-out read_speech', per_cell=PER_CELL, seed=20260818,
    )
    aishell_index = generate_controlled_benchmark(
        aishell_sources, noise_by_kind, real_rir_buckets, AISHELL_CER_DIR,
        dataset_id='aishell1_controlled_v1', purpose='controlled_chinese_cer',
        source_dataset='AISHELL-1 test', per_cell=PER_CELL, seed=20260819,
    )

wenet_index = generate_real_cer_benchmark(
    wns_sources, WENET_REAL_DIR, per_duration_bucket=WENET_PER_BUCKET,
)

if not QUICK_TEST:
    print(f'DNS 客观质量集: {len(dns_index["records"])} 条 → {DNS_QUALITY_DIR}')
    print(f'AISHELL 受控 CER: {len(aishell_index["records"])} 条 → {AISHELL_CER_DIR}')
print(f'WenetSpeech 真实 CER: {len(wenet_index["records"])} 条 → {WENET_REAL_DIR}')


### 结构性校验

这里不再用 VAD 猜“音频是否被截断”。源语音只按时长筛选、从不裁剪；验证重点是：

1. 受控集确实有80个噪声/RIR格 + 1个identity格，RT60没有再被折叠；
2. 标称、内存实测和写盘后 SNR 对齐；
3. 真实 RIR 的实测 RT60 落在对应桶内；
4. WenetSpeech 没有 `clean` 字段、没有二次 SNR/RIR 字段。


In [ ]:
import collections, statistics as st
from rtse.audio.io import read_audio
from rtse.data.synth import speech_active_mask

def measured_disk_snr(root, record):
    target = read_audio(Path(root, record['clean']))
    noisy = read_audio(Path(root, record['noisy']))
    signal = target - target.mean()
    noise = noisy - target
    noise = noise - noise.mean()
    mask = speech_active_mask(signal)
    return 10 * np.log10(np.mean(signal[mask] ** 2) / np.mean(noise ** 2))

def validate_controlled(path, expected_purpose):
    idx = json.loads(Path(path, 'index.json').read_text(encoding='utf-8'))
    recs = idx['records']
    assert idx['purpose'] == expected_purpose
    assert idx['reference_is_clean'] is True
    assert idx['strata'] == ['snr', 'noise_kind', 'rir_kind', 'rt60_bucket']
    assert len(recs) == (5 * 2 * 2 * 4 + 1) * PER_CELL
    cells = collections.Counter(tuple(r[k] for k in idx['strata']) for r in recs)
    assert len(cells) == 81 and set(cells.values()) == {PER_CELL}
    mixed = [r for r in recs if r['noise_kind'] != 'none']
    identity = [r for r in recs if r['noise_kind'] == 'none']
    assert len(identity) == PER_CELL and all(r['noisy'].endswith('_input.wav') for r in identity)
    worst_snr = max(abs(r['snr_measured'] - r['snr']) for r in mixed)
    assert worst_snr < 1.0, f'SNR 最大偏差 {worst_snr:.2f} dB'
    # 分层抽查写盘后的成对 WAV，锁定“输入/目标被独立归一化”这类隐蔽错误。
    stride = max(1, len(mixed) // 40)
    disk_probe = mixed[::stride][:40]
    worst_disk_snr = max(abs(measured_disk_snr(path, r) - r['snr']) for r in disk_probe)
    assert worst_disk_snr < 0.25, f'写盘后 SNR 最大偏差 {worst_disk_snr:.2f} dB'
    ranges = {t: (lo, hi) for t, lo, hi in RT60_BINS}
    for r in recs:
        if r['rir_kind'] == 'real':
            lo, hi = ranges[r['rt60_bucket']]
            assert lo <= r['rt60_measured'] < hi
    print(
        f'✓ {idx["dataset_id"]}: {len(recs)} 条 / 81格，'
        f'内存SNR最大偏差 {worst_snr:.2f} dB，写盘抽查 {worst_disk_snr:.2f} dB'
    )

if not QUICK_TEST:
    validate_controlled(DNS_QUALITY_DIR, 'objective_audio_quality')
    validate_controlled(AISHELL_CER_DIR, 'controlled_chinese_cer')

real = json.loads(Path(WENET_REAL_DIR, 'index.json').read_text(encoding='utf-8'))
assert real['reference_is_clean'] is False and real['cer_upper_is_meaningful'] is False
assert all('clean' not in r and 'snr' not in r and 'rir_kind' not in r for r in real['records'])
counts = collections.Counter(r['duration_bucket'] for r in real['records'])
assert counts == {'short': WENET_PER_BUCKET, 'medium': WENET_PER_BUCKET, 'long': WENET_PER_BUCKET}
print(f'✓ WenetSpeech 原始会议集: {len(real["records"])} 条，时长分桶 {dict(counts)}')


## 5. 打包三套测试集，下载到本地

解压后应得到 `data/testsets/{dns_objective,aishell_controlled,wenetspeech_real}`。


In [ ]:
!cd "{OUT}" && rm -f testsets_v1.zip && zip -q -r testsets_v1.zip testsets && ls -lh testsets_v1.zip
print()
print('下一步：')
print('  1. 继续跑 02_train.ipynb（SMOKE_RUN=True 先验证闭环）')
print(f'  2. 下载 {OUT}/testsets_v1.zip，解压到本地 data/ 下')
print('  3. 本地分别运行：')
print('     uv run rtse-eval data/testsets/dns_objective --skip-cer')
print('     uv run rtse-eval data/testsets/aishell_controlled')
print('     uv run rtse-eval data/testsets/wenetspeech_real --skip-objective')


---

# ═══════════════════════════════════════════════════════════════════
# PART 2 · 模型训练
# ═══════════════════════════════════════════════════════════════════

在线动态混音训练 CRN，跑完后过**无害性与有效性闸门**。

⚠️ 闸门不过就不要往下走 PART 3：导出一个连干净输入都会破坏的模型没有意义。

## 1. 构建数据集

Colab 每次连接给的都是**全新虚拟机**，本地盘是空的 —— 哪怕几分钟前刚在 01 里
跑完下载。下面会用 Drive 上缓存的压缩包重新解压（几分钟），压缩包也不在了才重下。
**这是正常现象，不用手动跳回 01。**

In [ ]:
mf = Path(MANIFEST)
assert mf.exists(), f'找不到 {mf}，先跑 01_data_prep.ipynb'
manifest = json.loads(mf.read_text(encoding='utf-8'))
QUICK_TEST = manifest['quick_test']
print(f'清单版本: {manifest.get("version")}   quick_test={QUICK_TEST}')
print(f'  语音 train/val: {len(manifest["speech"]["train"])}/{len(manifest["speech"]["val"])}')
print(f'  噪声 稳态/非稳态(train): {len(manifest["noise_stationary_train"])}/'
      f'{len(manifest["noise_nonstationary_train"])}')
print(f'  真实 RIR train: {len(manifest["rir_train"])}')

In [ ]:
from torch.utils.data import DataLoader
from rtse.data.dataset import OnlineMixDataset, MixConfig
from rtse.metrics.intrusive import si_sdr

# OnlineMixDataset 接受**文件列表**。必须用清单而不是让它扫目录 ——
# 清单是按说话人划分好的，扫目录会把验证说话人混进训练集，指标全部虚高。
# 用 MixConfig 的**默认值**，不要在这里写死 snr_range ——
# 默认值刚修过（上限 20→35 dB、恒等样本 2%→10%）：原来模型从没见过
# "已经干净"的输入，学成了无条件过抑制，下游 CER 在高 SNR 段大幅恶化。
# 在这里覆盖默认值 = 把那个 bug 请回来（见 docs/FINDINGS.md F-11）。
MIX = MixConfig(segment_seconds=4.0)

train_ds = OnlineMixDataset(manifest['speech']['train'], manifest['noise_train'],
                            manifest['rir_train'], cfg=MIX, length=2000 if SMOKE_RUN else 20000, seed=0)
val_ds = OnlineMixDataset(manifest['speech']['val'], manifest['noise_test'],
                          manifest['rir_test'], cfg=MIX, length=200 if SMOKE_RUN else 800, seed=999)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

nb_, cl_ = next(iter(train_dl))
print('batch', tuple(nb_.shape), '| 输入 SI-SDR 抽样:',
      [round(si_sdr(cl_[i].numpy(), nb_[i].numpy()), 1) for i in range(4)], 'dB')

## 2. 训练

**别照搬预估时间，看第一个 epoch 的实测值。** 训练器把每个 epoch 的
`epoch_seconds` 写进 `history.json`，第一个 epoch 跑完就能算出总时长。

第一次建议把 `EPOCHS` 改成 5 跑一轮，确认 loss 在降、时间可接受，再改回 60。
先跑 `crn-nano`（参数量只有 lite 的 1/5），它跑完就能拿到一套完整的端到端指标，
把整个流程闭环；大模型再慢慢跑。**有一个能用的模型，远好过两个都卡在半路。**

In [ ]:
from rtse.models import build_model
from rtse.train import Trainer, TrainConfig

MODELS = ['crn-nano'] if SMOKE_RUN else ['crn-nano', 'crn-lite']
EPOCHS = 3 if SMOKE_RUN else 60

for name in MODELS:
    out_dir = f'{CKPT_DIR}/{name}'     # ← 落在 Drive 上，会话断了也在
    os.makedirs(out_dir, exist_ok=True)
    cfg = TrainConfig(model=name, epochs=EPOCHS, batch_size=16, lr=3e-4,
                      out_dir=out_dir, num_workers=2, log_every=100)
    model = build_model(name)
    print(f'\n{"="*70}\n{name}   参数量 {model.count_params():,}   → {out_dir}\n{"="*70}')

    tr = Trainer(model, train_dl, val_dl, cfg)
    last = Path(out_dir) / 'last.pt'
    if last.exists():
        tr.load(last)                  # 断点续训（含优化器/调度/随机数状态）
    if tr.epoch >= EPOCHS:
        print(f'{name} 已完成（epoch {tr.epoch}），跳过'); continue
    tr.fit()

print('\n训练产物（在 Drive 上）：')
!ls -lh {shq(CKPT_DIR)}/*/

## 3. 无害性与有效性闸门

训练loss下降不等于模型可部署。导出前同时检查：

- clean→clean：完全干净输入不应再次被重度处理；
- noisy→target：留出混音的SI-SDR应有正增益。

冒烟版只打印诊断；正式版若干净透传中位SI-SDR低于20 dB或去噪增益不为正，
直接停止，不把失败模型导出成“最终模型”。


In [ ]:
import numpy as np
import torch
from rtse.audio.io import read_audio
from rtse.data.dataset import stft_torch, istft_torch
from rtse.metrics.intrusive import si_sdr

@torch.inference_mode()
def run_wave(model, wave):
    x = torch.as_tensor(wave, dtype=torch.float32, device=next(model.parameters()).device)[None]
    spec = stft_torch(x)
    out_spec = model(spec)
    return istft_torch(out_spec, length=x.shape[-1])[0].cpu().numpy()

gate_results = {}
for name in MODELS:
    model = build_model(name).to('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(f'{CKPT_DIR}/{name}/best.pt', map_location=next(model.parameters()).device,
                    weights_only=False)
    model.load_state_dict(ck['model'])
    model.eval()

    clean_scores = []
    for path in manifest['speech']['val'][:10]:
        wave = read_audio(path)
        if wave.size < 16000:
            continue
        wave = wave[:16000 * 4]
        clean_scores.append(si_sdr(wave, run_wave(model, wave)))

    gains = []
    for i in range(10):
        noisy, target = val_ds[i]
        noisy_np, target_np = noisy.numpy(), target.numpy()
        gains.append(si_sdr(target_np, run_wave(model, noisy_np)) - si_sdr(target_np, noisy_np))

    clean_median = float(np.median(clean_scores))
    gain_median = float(np.median(gains))
    gate_results[name] = {'clean_passthrough_si_sdr': clean_median,
                          'noisy_delta_si_sdr': gain_median}
    print(f'{name}: clean透传 {clean_median:.2f} dB；留出混音 ΔSI-SDR {gain_median:+.2f} dB')

    if not SMOKE_RUN:
        assert clean_median >= 20.0, f'{name} 对干净输入仍然过抑制，禁止导出'
        assert gain_median > 0.0, f'{name} 留出混音没有正增益，禁止导出'

Path(GATES).write_text(
    json.dumps(gate_results, ensure_ascii=False, indent=1), encoding='utf-8')


## 4. 训练曲线


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in MODELS:
    h = Path(f'{CKPT_DIR}/{name}/history.json')
    if not h.exists(): continue
    hist = json.loads(h.read_text())
    ep = [r['epoch'] for r in hist]
    axes[0].plot(ep, [r.get('loss') for r in hist], label=name)
    axes[1].plot(ep, [r.get('si_sdr') for r in hist], label=f'{name} train')
    if 'val_si_sdr' in hist[0]:
        axes[1].plot(ep, [r.get('val_si_sdr') for r in hist], '--', label=f'{name} val')
    axes[2].plot(ep, [r.get('spec') for r in hist], label=name)

for ax, t in zip(axes, ['总损失', 'SI-SDR (dB)', '压缩谱损失']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 怎么读：
#   训练 SI-SDR 一路涨但验证走平 → 过拟合，加数据或加正则
#   两条都走平且数值低         → 欠拟合，或学习率有问题

---

# ═══════════════════════════════════════════════════════════════════
# PART 3 · ONNX 导出 + 评测 + 打包回传
# ═══════════════════════════════════════════════════════════════════

流式 ONNX 导出与三层校验、DNS 客观质量评测（含 PESQ/DNSMOS），
最后打包成一个 `rtse_handoff.zip` 供下载。

## 1. 导出并校验

In [ ]:
# 正式导出必须先通过 PART 2 的 clean透传 + 去噪正增益闸门。
gate_path = Path(GATES)
assert gate_path.exists(), '缺少 training_gates.json；先完整运行 PART 2 的无害性与有效性闸门'
training_gates = json.loads(gate_path.read_text(encoding='utf-8'))
if not SMOKE_RUN:
    for name, gate in training_gates.items():
        assert gate['clean_passthrough_si_sdr'] >= 20.0, f'{name} 干净透传未达标，禁止正式导出'
        assert gate['noisy_delta_si_sdr'] > 0.0, f'{name} 去噪无正增益，禁止正式导出'
print('训练闸门:', training_gates)

import torch
from rtse.models import build_model
from rtse.train.export import export_streaming_onnx

MODELS = ['crn-nano'] if SMOKE_RUN else ['crn-nano', 'crn-lite']
export_info = {}

for name in MODELS:
    ck = Path(f'{CKPT_DIR}/{name}/best.pt')
    if not ck.exists():
        print(f'[skip] {name}: 没有 best.pt，先跑 02_train.ipynb'); continue

    state = torch.load(ck, map_location='cpu', weights_only=False)
    model = build_model(name)
    model.load_state_dict(state['model'])
    model.eval()

    info = export_streaming_onnx(model, f'{MODEL_DIR}/{name}.onnx', verify=True)
    v = info['verification']
    export_info[name] = info

    print(f'\n=== {name} ===')
    print(f"  训练到 epoch {state['epoch']}，最佳 val SI-SDR {-state['best_val']:.3f} dB")
    print(f"  参数 {info['params']:,}   文件 {info['size_kb']} KB")
    print(f"  ONNX流式 vs PyTorch整段 : {v['onnx_vs_pytorch_batch']:.3e} (相对 {v['relative_error']:.3e})")
    print(f"  PyTorch流式 vs 整段     : {v['pytorch_streaming_vs_batch']:.3e}")
    print(f"  状态形状稳定            : {v['state_shape_stable']}")
    print(f"  ==> {'PASS ✅' if v['passed'] else 'FAIL ❌ 不要下载这个模型，先查因果性'}")

Path(f'{MODEL_DIR}/export_info.json').write_text(
    json.dumps(export_info, ensure_ascii=False, indent=1), encoding='utf-8')


## 2. 下载 DNSMOS 模型

DNSMOS 是**无参考** MOS 预测，是实时麦克风演示里唯一能显示的质量指标
（那里没有干净参考）。模型来自微软 DNS-Challenge 仓库，只有几 MB。

In [ ]:
os.makedirs(f'{MODEL_DIR}/dnsmos', exist_ok=True)
!wget -q -O "{MODEL_DIR}/dnsmos/sig_bak_ovr.onnx" \
  https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx \
  && ls -lh "{MODEL_DIR}/dnsmos/"

# 若 404，去 https://github.com/microsoft/DNS-Challenge 的 DNSMOS 目录确认最新路径。
# 拿不到也不影响主流程：本地评测会自动把 DNSMOS 列标 n/a。

## 3. 在 DNS 客观质量集上评测（含 PESQ）

这是唯一负责 SI-SDR/STOI/PESQ 的测试集。按噪声平稳性、RIR来源和匹配后的
RT60 桶汇总；中文测试集不在这里冒充干净参考。


In [ ]:
import numpy as np
from tqdm.auto import tqdm
from rtse.audio.io import read_audio
from rtse.metrics.intrusive import si_sdr, stoi, estoi, pesq, seg_snr
from rtse.runtime import Pipeline, OnnxEnhancer
from rtse.dsp import build_dsp
from rtse.vad import build_vad

idx = json.loads(Path(f'{DNS_QUALITY_DIR}/index.json').read_text(encoding='utf-8'))
records = idx['records']
print(f'测试集 {len(records)} 个样本')

METHODS = ['none', 'specsub', 'wiener', 'mmse-lsa'] + list(export_info)

def make(method):
    if method == 'none': return None
    if method in ('specsub', 'wiener', 'mmse-lsa'): return build_dsp(method)
    return OnnxEnhancer(f'{MODEL_DIR}/{method}.onnx')

rows = []
for method in METHODS:
    enh = make(method)
    pipe = Pipeline(enhancer=enh, vad=build_vad('energy'))
    for r in tqdm(records, desc=f'{method:>10}', leave=False):
        clean = read_audio(f'{DNS_QUALITY_DIR}/{r["clean"]}')
        noisy = read_audio(f'{DNS_QUALITY_DIR}/{r["noisy"]}')
        pipe.reset()
        out, _ = pipe.process_signal(noisy)
        rows.append({'id': r['id'], 'method': method, 'snr': r['snr'],
                     'noise_kind': r['noise_kind'], 'rir_kind': r['rir_kind'],
                     'rt60_bucket': r['rt60_bucket'],
                     'rt60_measured': r['rt60_measured'],
                     'si_sdr': si_sdr(clean, out), 'seg_snr': seg_snr(clean, out),
                     'stoi': stoi(clean, out), 'estoi': estoi(clean, out),
                     'pesq': pesq(clean, out)})

Path(COLAB_METRICS).write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
print(f'已写入 {len(rows)} 条指标 → {COLAB_METRICS}')


In [ ]:
# 快速汇总；正式报告用本地 rtse-eval 的逐条 JSON。
import collections, statistics as st
print(f"{'method':<12}{'noise':<15}{'rir':<8}{'RT60':>7}{'SI-SDR':>9}{'STOI':>8}{'PESQ':>8}")
print('-' * 68)
agg = collections.defaultdict(list)
for r in rows:
    agg[(r['method'], r['noise_kind'], r['rir_kind'], r['rt60_bucket'])].append(r)
for key in sorted(agg, key=lambda x: tuple(str(v) for v in x)):
    m, nk, rk, rt = key
    g = agg[key]
    pq = [r['pesq'] for r in g if r['pesq'] is not None]
    print(f"{m:<12}{nk:<15}{rk:<8}{rt:>7.1f}"
          f"{st.mean(r['si_sdr'] for r in g):>9.2f}"
          f"{st.mean(r['stoi'] for r in g):>8.3f}"
          f"{(st.mean(pq) if pq else float('nan')):>8.3f}")


## 4. 打包回传

最后只需下载一个 `rtse_handoff.zip`。解压到新电脑的仓库根目录后，会直接得到：

- `models/`：本轮ONNX、导出验证信息、DNSMOS；
- `checkpoints/`：本轮模型的best/last/history，可续训；
- `data/testsets/`：三套固定测试集；
- `results/`：Colab客观指标、训练闸门、数据清单；
- `HANDOFF_MANIFEST.json`：每个文件的大小与SHA-256，供本地检查传输完整性。

不会包含几十GB的 `archives/` 原始下载缓存，也不会夹带旧模型。


In [ ]:
import datetime as dt
import hashlib
import zipfile

handoff = Path(WORK) / 'rtse_handoff'
if handoff.exists():
    shutil.rmtree(handoff)
(handoff / 'results').mkdir(parents=True)
(handoff / 'checkpoints').mkdir(parents=True)
(handoff / 'models').mkdir(parents=True)

required = {
    'models': Path(MODEL_DIR),
    'testsets': Path(TESTSETS_DIR),
    'colab_metrics': Path(COLAB_METRICS),
    'training_gates': Path(GATES),
    'data_manifest': Path(MANIFEST),
}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
assert not missing, '回传包缺少必要产物：\n' + '\n'.join(missing)

shutil.copytree(required['testsets'], handoff / 'data' / 'testsets')
shutil.copy2(required['colab_metrics'], handoff / 'results' / 'colab_metrics.json')
shutil.copy2(required['training_gates'], handoff / 'results' / 'training_gates.json')
shutil.copy2(required['data_manifest'], handoff / 'results' / 'manifest.json')

# 只打包本轮配置中实际训练的模型，防止Drive里残留的旧权重/checkpoint混入。
export_info = Path(MODEL_DIR) / 'export_info.json'
dnsmos_dir = Path(MODEL_DIR) / 'dnsmos'
assert export_info.exists(), '缺少ONNX导出验证信息 export_info.json'
assert (dnsmos_dir / 'sig_bak_ovr.onnx').exists(), '缺少DNSMOS模型'
shutil.copy2(export_info, handoff / 'models' / 'export_info.json')
shutil.copytree(dnsmos_dir, handoff / 'models' / 'dnsmos')
for name in MODELS:
    onnx_path = Path(MODEL_DIR) / f'{name}.onnx'
    assert onnx_path.exists(), f'{name} ONNX不存在'
    shutil.copy2(onnx_path, handoff / 'models' / onnx_path.name)
    src = Path(CKPT_DIR) / name
    assert (src / 'best.pt').exists() and (src / 'last.pt').exists(), f'{name} checkpoint不完整'
    shutil.copytree(src, handoff / 'checkpoints' / name)

files = sorted(p for p in handoff.rglob('*') if p.is_file())
handoff_meta = {
    'schema_version': 1,
    'created_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'smoke_run': SMOKE_RUN,
    'models': list(MODELS),
    'file_count_without_manifest': len(files),
    'files': [
        {
            'path': p.relative_to(handoff).as_posix(),
            'bytes': p.stat().st_size,
            'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
        }
        for p in files
    ],
}
(handoff / 'HANDOFF_MANIFEST.json').write_text(
    json.dumps(handoff_meta, ensure_ascii=False, indent=1), encoding='utf-8')

archive = Path(OUT) / 'rtse_handoff.zip'
archive.unlink(missing_ok=True)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in sorted(p for p in handoff.rglob('*') if p.is_file()):
        zf.write(path, Path('rtse_handoff') / path.relative_to(handoff))

print(f'✓ 完整回传包: {archive}')
print(f'  文件 {len(files) + 1} 个，压缩后 {archive.stat().st_size / 2**20:.1f} MB')
print('下载这一个ZIP即可；解压后把 rtse_handoff/ 内各目录合并到仓库根目录。')
